# 🚬 흡연 분류 - V7 BEST (최종)

**V3 장점 + V7 장점 결합**
- ✅ RandomizedSearchCV 튜닝
- ✅ OOF 기반 검증
- ✅ CatBoost 범주형 처리
- ✅ 핵심 파생 피처만
- ✅ 클리핑 OFF

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

# 경로
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

print("✅ 준비 완료!")

In [ ]:
# 데이터 로드
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"컬럼: {train.columns.tolist()}")

In [ ]:
# 한글 → 영어 컬럼 매핑
def map_columns(df):
    df = df.copy()
    mapping = {}
    cat_cols = []
    
    for col in df.columns:
        c = col.lower()
        if 'id' in c:
            mapping[col] = 'id'
        elif '나이' in col or 'age' in c:
            mapping[col] = 'age'
        elif '키' in col or 'height' in c:
            mapping[col] = 'height'
        elif '몸무게' in col or 'weight' in c:
            mapping[col] = 'weight'
        elif 'bmi' in c:
            mapping[col] = 'bmi'
        elif '시력' in col:
            mapping[col] = 'eyesight'
        elif '충치' in col:
            mapping[col] = 'cavity'
            cat_cols.append('cavity')
        elif '혈당' in col or '공복' in col:
            mapping[col] = 'fasting_blood_sugar'
        elif '혈압' in col:
            mapping[col] = 'blood_pressure'
        elif '중성' in col:
            mapping[col] = 'triglyceride'
        elif '크레' in col:
            mapping[col] = 'serum_creatinine'
        elif '콜레스테롤' in col:
            mapping[col] = 'cholesterol'
        elif '고밀도' in col:
            mapping[col] = 'hdl'
        elif '저밀도' in col:
            mapping[col] = 'ldl'
        elif '헤모글로빈' in col:
            mapping[col] = 'hemoglobin'
        elif '단백' in col and '지단백' not in col:
            mapping[col] = 'urine_protein'
            cat_cols.append('urine_protein')
        elif '간' in col or '효소' in col:
            mapping[col] = 'gtp'
        elif 'label' in c:
            mapping[col] = 'label'
        else:
            mapping[col] = col.lower().replace(' ', '_').replace('(', '').replace(')', '')
    
    return df.rename(columns=mapping), cat_cols

train_df, cat_cols = map_columns(train)
test_df, _ = map_columns(test)

print(f"매핑 후 컬럼: {train_df.columns.tolist()}")
print(f"범주형: {cat_cols}")

In [ ]:
# ID 제거, X/y 분리
if 'id' in train_df.columns:
    train_df = train_df.drop('id', axis=1)
    test_df = test_df.drop('id', axis=1)

X = train_df.drop('label', axis=1)
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

print(f"X: {X.shape}, y: {y.shape}")
print(f"흡연자 비율: {y.mean()*100:.1f}%")

In [ ]:
# 핵심 파생 피처 (최소한만)
def create_features(df, cat_cols):
    df = df.copy()
    cols = df.columns.tolist()
    
    # 1. TG/HDL 비율
    if 'triglyceride' in cols and 'hdl' in cols:
        df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
    
    # 2. 헤모글로빈 높음 (범주형)
    if 'hemoglobin' in cols:
        df['hemo_high'] = (df['hemoglobin'] > 15.5).astype(int)
        cat_cols.append('hemo_high')
    
    # 3. GTP 높음 (범주형)
    if 'gtp' in cols:
        df['gtp_high'] = (df['gtp'] > 40).astype(int)
        cat_cols.append('gtp_high')
    
    # 4. 나이대 (범주형)
    if 'age' in cols:
        df['age_group'] = pd.cut(df['age'], bins=[0,35,45,55,100], labels=[0,1,2,3]).astype(int)
        cat_cols.append('age_group')
    
    # 5. 헤모글로빈 × GTP
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['hemo_x_gtp'] = df['hemoglobin'] * df['gtp']
    
    return df.fillna(0), list(set(cat_cols))

X_fe, cat_cols = create_features(X, cat_cols)
X_test_fe, _ = create_features(X_test, [])

print(f"피처 수: {X.shape[1]} → {X_fe.shape[1]}")
print(f"범주형 컬럼: {cat_cols}")

In [ ]:
# 클래스 비율
scale_pos = (y == 0).sum() / (y == 1).sum()
print(f"클래스 비율: {scale_pos:.2f}")

# CatBoost용 범주형 컬럼명
cat_features = [c for c in cat_cols if c in X_fe.columns]
print(f"CatBoost 범주형: {cat_features}")

## 🔧 하이퍼파라미터 튜닝

In [ ]:
print("=" * 50)
print("🔧 XGBoost 튜닝")
print("=" * 50)

xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'min_child_weight': [1, 3, 5],
    'scale_pos_weight': [1, scale_pos]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='error'),
    xgb_params, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_fe.values, y)
best_xgb = xgb_search.best_params_
print(f"\n✅ XGBoost: {xgb_search.best_score_:.5f}")

In [ ]:
print("=" * 50)
print("🔧 LightGBM 튜닝")
print("=" * 50)

lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'class_weight': ['balanced', None]
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_fe.values, y)
best_lgb = lgb_search.best_params_
print(f"\n✅ LightGBM: {lgb_search.best_score_:.5f}")

In [ ]:
print("=" * 50)
print("🔧 CatBoost 튜닝")
print("=" * 50)

cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5],
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0, cat_features=cat_features, auto_class_weights='Balanced'),
    cat_params, n_iter=40, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X_fe, y)
best_cat = cat_search.best_params_
print(f"\n✅ CatBoost: {cat_search.best_score_:.5f}")

In [ ]:
print("\n" + "=" * 50)
print("📊 튜닝 결과")
print("=" * 50)
print(f"XGBoost:  {xgb_search.best_score_:.5f}")
print(f"LightGBM: {lgb_search.best_score_:.5f}")
print(f"CatBoost: {cat_search.best_score_:.5f}")

## 🎯 OOF 예측 (튜닝된 파라미터)

In [ ]:
print("=" * 50)
print("🎯 5-Fold OOF 예측")
print("=" * 50)

N_SPLITS = 5
SEED = 42

oof_xgb = np.zeros(len(X_fe))
oof_lgb = np.zeros(len(X_fe))
oof_cat = np.zeros(len(X_fe))

test_xgb = np.zeros(len(X_test_fe))
test_lgb = np.zeros(len(X_test_fe))
test_cat = np.zeros(len(X_test_fe))

kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(kfold.split(X_fe, y)):
    print(f"\nFold {fold+1}/{N_SPLITS}")
    
    X_tr, X_va = X_fe.iloc[tr_idx], X_fe.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
    # XGBoost
    xgb_m = XGBClassifier(**best_xgb, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='error')
    xgb_m.fit(X_tr.values, y_tr)
    oof_xgb[va_idx] = xgb_m.predict_proba(X_va.values)[:, 1]
    test_xgb += xgb_m.predict_proba(X_test_fe.values)[:, 1] / N_SPLITS
    
    # LightGBM
    lgb_m = LGBMClassifier(**best_lgb, random_state=SEED, verbose=-1)
    lgb_m.fit(X_tr.values, y_tr)
    oof_lgb[va_idx] = lgb_m.predict_proba(X_va.values)[:, 1]
    test_lgb += lgb_m.predict_proba(X_test_fe.values)[:, 1] / N_SPLITS
    
    # CatBoost (DataFrame)
    cat_m = CatBoostClassifier(**best_cat, random_state=SEED, verbose=0, cat_features=cat_features, auto_class_weights='Balanced')
    cat_m.fit(X_tr, y_tr)
    oof_cat[va_idx] = cat_m.predict_proba(X_va)[:, 1]
    test_cat += cat_m.predict_proba(X_test_fe)[:, 1] / N_SPLITS
    
    print(f"  XGB: {accuracy_score(y_va, (oof_xgb[va_idx]>=0.5).astype(int)):.4f}")
    print(f"  LGB: {accuracy_score(y_va, (oof_lgb[va_idx]>=0.5).astype(int)):.4f}")
    print(f"  CAT: {accuracy_score(y_va, (oof_cat[va_idx]>=0.5).astype(int)):.4f}")

print("\n✅ OOF 완료!")
print(f"\n전체 OOF Accuracy:")
print(f"  XGB: {accuracy_score(y, (oof_xgb>=0.5).astype(int)):.5f}")
print(f"  LGB: {accuracy_score(y, (oof_lgb>=0.5).astype(int)):.5f}")
print(f"  CAT: {accuracy_score(y, (oof_cat>=0.5).astype(int)):.5f}")

## 🔍 가중치 + 임계값 최적화

In [ ]:
print("=" * 50)
print("🔍 가중치 최적화")
print("=" * 50)

best_score = 0
best_w = (0.33, 0.33, 0.34)

for w1 in np.arange(0.1, 0.7, 0.05):
    for w2 in np.arange(0.1, 0.7, 0.05):
        w3 = round(1 - w1 - w2, 2)
        if w3 >= 0.1:
            blend = w1*oof_xgb + w2*oof_lgb + w3*oof_cat
            acc = accuracy_score(y, (blend >= 0.5).astype(int))
            if acc > best_score:
                best_score = acc
                best_w = (w1, w2, w3)

w1, w2, w3 = best_w
print(f"\n🏆 최적 가중치: XGB={w1:.2f}, LGB={w2:.2f}, CAT={w3:.2f}")
print(f"   Accuracy: {best_score:.5f}")

In [ ]:
print("\n" + "=" * 50)
print("🔍 임계값 최적화")
print("=" * 50)

oof_blend = w1*oof_xgb + w2*oof_lgb + w3*oof_cat

best_th = 0.5
best_acc = 0

for th in np.arange(0.35, 0.65, 0.01):
    acc = accuracy_score(y, (oof_blend >= th).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_th = th

print(f"\n🏆 최적 임계값: {best_th:.2f}")
print(f"   OOF Accuracy: {best_acc:.5f}")

## 📝 제출 파일 생성

In [ ]:
# 최종 예측
test_blend = w1*test_xgb + w2*test_lgb + w3*test_cat
final_pred = (test_blend >= best_th).astype(int)

print(f"예측 분포:")
print(f"  비흡연(0): {(final_pred==0).sum()}명 ({(final_pred==0).mean()*100:.1f}%)")
print(f"  흡연(1):   {(final_pred==1).sum()}명 ({(final_pred==1).mean()*100:.1f}%)")

In [ ]:
# 저장
sub = submission.copy()
sub['label'] = final_pred
sub['label'] = sub['label'].astype(int)

output = result_path + 'submission_v7_best.csv'
sub.to_csv(output, index=False)

print(f"✅ 저장: {output}")
display(sub.head())

In [ ]:
from google.colab import files
files.download(output)

print("\n" + "=" * 60)
print("🎉 V7 BEST 완료!")
print("=" * 60)
print(f"\n📊 최종 설정:")
print(f"   가중치: XGB={w1:.2f}, LGB={w2:.2f}, CAT={w3:.2f}")
print(f"   임계값: {best_th:.2f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"\n🚀 제출하세요!")